# CIS 5450 Project: Difficulty Topics
**Team Members:** Yifan Cai, Yuanqi Zhen, Yiran Hu, Shanae Obi

> This notebook documents how you implemented difficulty topics in your project. Use the link button in the top right when you select a cell to get a **hyperlink**.


---

## Topic: 1: **Feature Engineering**
**Hyperlink:**
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html


**Implementation & Rationale**

We design a multi-stage (3 stage) feature engineering pipeline that transforms raw event-level interaction logs into structured user-level representations suitable for machine learning models.

Starting from approximately 40 million raw interaction records, we construct an event-derived feature table (event_feature_table_v3) that captures multiple dimensions of user behavior. These include behavioral intensity (e.g., total_events, total_carts), temporal dynamics (e.g., activity duration and inter-event time gaps), session structure (e.g., number of sessions, session length statistics), and preference/diversity signals such as category entropy and top category.

In addition, we incorporate interaction-level patterns such as funnel transitions (e.g., view_to_cart_rate) and repeated engagement signals (e.g., repeated views of the same product). Price-related features are also included to capture user sensitivity to product price ranges.

A key design consideration is converting variable-length user interaction sequences into fixed-length feature vectors, enabling compatibility with traditional models such as Linear Regression and XGBoost. Furthermore, we introduce categorical representations using one-hot encoding for linear and tree-based models, and learnable embeddings for the MLP model, allowing the network to capture relationships between product categories.

Compared to simpler approaches such as using only aggregated counts, this design allows us to explicitly capture temporal ordering and behavioral structure, which are critical for modeling user intent. Directly using raw event sequences is not feasible for traditional models, making this transformation necessary.

**Evidence of Effectiveness**

Empirically, we observe that adding event-derived features (Step 2) leads to the largest performance improvement across all models, particularly in F1 and PR-AUC. This indicates that temporal and behavioral patterns extracted from raw interaction logs provide substantial predictive signal beyond simple aggregated statistics.

Additionally, embedding-based representations in the MLP model (Step 3) further improve performance, demonstrating that richer feature representations—especially those capturing category relationships—can enhance model capacity.

However, the improvement from Step 3 is model-dependent. While embedding features significantly benefit the MLP model, they provide limited gains for linear and tree-based models. This suggests that the effectiveness of richer feature representations depends on the model's capacity to capture complex feature interactions.

---

## Topic: 2: **Imbalance Data Handling**
**Hyperlink:** https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html

https://xgboost.readthedocs.io/en/stable/parameter.html

https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html

**Implementation & Rationale**

The dataset exhibits significant class imbalance, with only approximately 11.5% of users making purchases. This creates a challenge where naive models may achieve high accuracy while failing to correctly identify high-intent users.

To address this, we adopt a multi-level imbalance handling strategy.

At the data level, we use stratified train–test splitting to preserve the class distribution between training and testing sets, ensuring consistent evaluation.

At the model level, we apply cost-sensitive learning. For XGBoost, we set scale_pos_weight as the ratio of negative to positive samples, increasing the contribution of minority-class instances during training. For the MLP model, we use weighted binary cross-entropy (BCEWithLogitsLoss with pos_weight), penalizing misclassification of positive samples more heavily.

At the evaluation level, we adopt imbalance-aware metrics such as F1-score, PR-AUC, and MCC, which better reflect performance on the minority class compared to accuracy.

In addition, we perform threshold tuning on the MLP model. By increasing the decision threshold to around 0.7, the model becomes more conservative, significantly improving precision while maintaining stable F1. This demonstrates an effective trade-off between false positives and recall.

We do not apply resampling methods such as SMOTE or upsampling. This is because our feature space consists of aggregated behavioral representations, where synthetic interpolation may generate unrealistic user patterns and distort the original data distribution.

**Evidence of Effectiveness**

With these strategies, the models achieve strong performance on imbalance-sensitive metrics such as PR-AUC and F1-score. In particular, threshold tuning allows the model to better align predictions with high-intent users, improving practical usefulness in downstream applications such as targeting.


---

## Topic: 3: **Feature Importance**
**Hyperlink:**
https://xgboost.readthedocs.io/en/release_2.0.0/python/python_api.html

**Implementation & Rationale**

We analyze feature importance using model-specific methods to better understand which behavioral signals drive purchase prediction.

For linear models, we examine feature coefficients from Linear Regression to understand the influence of each predictor. In addition, we use tree-based models such as XGBoost, where feature importance scores (e.g., feature_importances_) provide a ranking of predictors based on their contribution to model decisions. For the MLP model with embedding, feature importance is not directly interpretable in the same way as linear or tree-based models. Instead, the model learns latent representations through non-linear transformations and embedding layers, which implicitly capture feature interactions and category relationships.

In particular, the embedding layer allows the model to represent categorical features in a continuous space, enabling it to learn similarities between categories and their impact on purchase behavior. This provides additional modeling flexibility beyond explicit feature importance rankings.

This dual approach allows us to capture both linear relationships and non-linear feature interactions, providing a more comprehensive understanding of feature relevance.

We prefer model-based importance methods over filter-based approaches because they reflect how features are actually used by the model during prediction, rather than relying solely on statistical relationships.

**Evidence of Effectiveness**

Across models, we observe that cart-related features (e.g., total_carts, has_carted, view_to_cart_rate) consistently rank among the most important predictors. This aligns with domain intuition, as cart interactions indicate strong purchase intent.

Furthermore, repeated engagement signals, session-level features, and category diversity also contribute meaningfully, suggesting that both immediate intent signals and longer-term behavioral patterns are important for predicting conversion.

These findings also explain why Step 2 (event-derived features) leads to the largest performance gain, as these features capture the most informative behavioral signals. In the MLP model, the performance improvement observed in Step 3 suggests that the embedding-based representation is effectively capturing additional information that is not fully utilized by linear or tree-based models.

However, feature importance should be interpreted with caution. For example, correlated features may share importance, and tree-based importance scores may be biased toward high-cardinality features. Despite these limitations, the consistency of results across models strengthens our conclusions.